# Spike A — RAG-MCP Tool Selection (DevOps Latency Investigation)

Companion notebook to `skills/tool-orchestration/rag-mcp-tool-selection/`.

**Scenario (from Agentic Graph RAG Ch5/Ch6 running example):** the DevOps agent investigates a latency spike in the checkout API. The AWS account is fictional (`123456789012`). All boto3 calls go through `moto` mocks so the notebook runs against zero real AWS credentials.

**What this spike validates:**
1. Chapter primitive → SKILL.md round-trip
2. Hand-rolled Python CLI is harness-portable
3. RAG-MCP filter produces the book's claimed token reduction (50-70%)
4. Real boto3 + `moto`-mocked AWS service returns shaped data

Per `PLAN.md` vertical-slice ordering, this is the integration spike. If it passes, the broader chapter conversion is low-risk repetition.

## 1. Load the skill

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SKILL_DIR = PROJECT_ROOT / 'skills' / 'tool-orchestration' / 'rag-mcp-tool-selection'
sys.path.insert(0, str(SKILL_DIR))

import lib
print(f'Loaded skill from: {SKILL_DIR}')
registry = lib.load_registry(SKILL_DIR / 'sample-aws-tools.json')
print(f'Registry has {len(registry)} AWS service tools.')

Loaded skill from: C:\Users\nnnnn\projects\agentic-graph-rag-skills\skills\tool-orchestration\rag-mcp-tool-selection
Registry has 30 AWS service tools.


## 2. The DevOps query

A real on-call engineer asks the agent:

In [2]:
QUERY = 'why is the checkout api slow during the 3pm latency spike'
print(QUERY)

why is the checkout api slow during the 3pm latency spike


## 3. Baseline — every tool dumped into the prompt

This is what the agent sees with the standard MCP `tools/list` operation. At 30 tools the prompt is already large; the book's Block / Goose anchor scales to 60+ MCP servers exposing thousands of tools.

In [3]:
baseline = lib.baseline_prompt(registry, QUERY)
baseline_tokens = lib.approximate_token_count(baseline)
print(f'Baseline prompt: {baseline_tokens} tokens, {len(baseline)} chars')
print('First 400 chars:')
print(baseline[:400] + '...')

Baseline prompt: 1149 tokens, 7512 chars
First 400 chars:
You are a DevOps agent. The user's query: 'why is the checkout api slow during the 3pm latency spike'

You may use the following tools (no filtering):

- cloudwatch_logs_insights_query: Run a CloudWatch Logs Insights query across one or more log groups to find errors, latency spikes, or specific patterns in service logs.
  parameters: logGroupNames=list[str], queryString=str (CloudWatch Logs Insig...


## 4. RAG-MCP filter — three-step pipeline

1. **Retrieve** — score every tool by word overlap against the query (production: swap for embeddings).
2. **Validate** — drop tools whose name and topics share no token with the query.
3. **Invoke** — format only the selected tools into the prompt.

In [4]:
result = lib.select(QUERY, SKILL_DIR / 'sample-aws-tools.json', top_k=5)
print(f"Registry size: {result['registry_size']} tools")
print(f"Selected: {len(result['selected'])} tools\n")
for s in result['selected']:
    print(f"  [{s['score']:.3f}]  {s['name']}")
    print(f"           {s['description']}")
print()
print(f"Baseline tokens: {result['baseline_tokens']}")
print(f"Filtered tokens: {result['filtered_tokens']}")
print(f"Reduction:       {result['reduction_pct']}%")

Registry size: 30 tools
Selected: 3 tools

  [0.471]  cloudwatch_logs_insights_query
           Run a CloudWatch Logs Insights query across one or more log groups to find errors, latency spikes, or specific patterns in service logs.
  [0.141]  rds_describe_db_log_files
           List RDS database log files available for download. Useful for inspecting slow-query logs, error logs, and general logs during database performance investigations.
  [0.120]  xray_get_trace_summaries
           Retrieve X-Ray distributed trace summaries to identify slow segments, error propagation paths, and service-to-service latency in a microservices architecture.

Baseline tokens: 1149
Filtered tokens: 179
Reduction:       84.4%


**Verification gate (per `PLAN.md`):** reduction should be > 50% for this 30-tool registry, matching the book's 50-70% claim. If we exceed that, the registry is small enough that the percentage is higher; if we're below, the retriever is broken.

In [5]:
assert result['reduction_pct'] > 50.0, (
    f"Token reduction {result['reduction_pct']}% below the 50% floor."
    " Check retriever or registry coverage."
)
print(f'PASS: {result["reduction_pct"]}% reduction (book floor: 50%, ceiling: 70%)')

PASS: 84.4% reduction (book floor: 50%, ceiling: 70%)


## 5. Real boto3 call against moto-mocked AWS

Now the agent acts on the top-1 selected tool. `cloudwatch_logs_insights_query` maps to `boto3.client('logs').start_query`. We mock the entire AWS surface with `moto` so the notebook runs against fictional account `123456789012` — same boto3 code, no credentials, no charges.

To target your real AWS account: remove the `@mock_aws` decorator and `os.environ['AWS_*']` setup.

In [6]:
import os
import time

# Fictional account placeholder — moto accepts any 12-digit account ID
FICTIONAL_ACCOUNT_ID = '123456789012'
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')
os.environ.setdefault('AWS_ACCESS_KEY_ID', 'testing')
os.environ.setdefault('AWS_SECRET_ACCESS_KEY', 'testing')

import boto3
from moto import mock_aws


@mock_aws
def investigate_checkout_latency():
    logs = boto3.client('logs')
    
    # Set up a log group with synthetic events that mirror a real latency spike
    log_group = '/aws/ecs/checkout-api'
    logs.create_log_group(logGroupName=log_group)
    logs.create_log_stream(logGroupName=log_group, logStreamName='task-abc123')
    
    now_ms = int(time.time() * 1000)
    events = [
        {'timestamp': now_ms - 600_000, 'message': 'INFO checkout request_id=req-001 duration_ms=42 status=200'},
        {'timestamp': now_ms - 300_000, 'message': 'WARN checkout request_id=req-007 duration_ms=2814 status=200 db_pool_wait_ms=2700'},
        {'timestamp': now_ms - 180_000, 'message': 'ERROR checkout request_id=req-013 duration_ms=5012 status=504 db_pool_exhausted=true'},
        {'timestamp': now_ms - 60_000,  'message': 'ERROR checkout request_id=req-019 duration_ms=4998 status=504 db_pool_exhausted=true'},
        {'timestamp': now_ms,           'message': 'WARN checkout request_id=req-024 duration_ms=3110 status=200 db_pool_wait_ms=2900'},
    ]
    logs.put_log_events(
        logGroupName=log_group,
        logStreamName='task-abc123',
        logEvents=events,
    )
    
    # The actual boto3 call the agent would make based on the top-1 selected tool
    # In production: parse the Logs Insights query language from the LLM's response
    start_response = logs.start_query(
        logGroupName=log_group,
        startTime=int(now_ms / 1000) - 3600,
        endTime=int(now_ms / 1000),
        queryString="""
            fields @timestamp, @message
            | filter status=504 or duration_ms > 1000
            | sort @timestamp desc
            | limit 20
        """,
    )
    return {
        'log_group': log_group,
        'query_id': start_response['queryId'],
        'events_seeded': len(events),
        'fictional_account': FICTIONAL_ACCOUNT_ID,
    }


boto3_result = investigate_checkout_latency()
print('boto3 + moto mock result:')
for k, v in boto3_result.items():
    print(f'  {k}: {v}')

boto3 + moto mock result:
  log_group: /aws/ecs/checkout-api
  query_id: 37a5b645-0289-1f1f-9341-5a0ef6e44fa3
  events_seeded: 5
  fictional_account: 123456789012


**Verification gate (per `PLAN.md`):** the mocked CloudWatch Logs Insights call must return shaped data — specifically a `queryId` that downstream code can poll. If the call shape is wrong, the spike fails on the AWS-seam axis.

In [7]:
assert isinstance(boto3_result['query_id'], str) and len(boto3_result['query_id']) > 0, (
    'CloudWatch Logs Insights start_query did not return a queryId.'
)
assert boto3_result['events_seeded'] == 5
print('PASS: boto3 + moto seam works. CloudWatch Logs Insights returned a queryId.')

PASS: boto3 + moto seam works. CloudWatch Logs Insights returned a queryId.


## 6. (Optional) Call Anthropic with the filtered prompt

If `ANTHROPIC_API_KEY` is set in your environment, the next cell sends the filtered prompt to Claude and prints which tool Claude picks. If the key is not set, the cell skips gracefully.

This is the practitioner-validation step: confirm that the RAG-MCP-filtered prompt is sufficient for Claude to choose the right tool without seeing the other 25 tools.

In [8]:
if not os.getenv('ANTHROPIC_API_KEY'):
    print('SKIPPED: set ANTHROPIC_API_KEY to run this cell against the real API.')
else:
    import anthropic
    client = anthropic.Anthropic()
    msg = client.messages.create(
        model='claude-sonnet-4-6',
        max_tokens=400,
        messages=[{'role': 'user', 'content': result['filtered_prompt']}],
    )
    print('Claude response:\n')
    for block in msg.content:
        if block.type == 'text':
            print(block.text)

Claude response:

## Analysis & Tool Selection Strategy

The user is reporting a **latency spike in the checkout API around 3PM**. To diagnose this, I need to understand *where* the slowness originates — it could be:

- **Application-level** slowness (slow code paths, timeouts, retries)
- **Downstream service** bottlenecks (payment service, inventory, auth)
- **Database** slow queries (RDS)
- **External dependency** latency

The best starting point is **distributed tracing via X-Ray**, because it will show me the *entire request lifecycle* across all services and pinpoint which **segment/subsegment** is the bottleneck — without me having to guess which log group or DB to check first.

---

## Tool Call #1: X-Ray Trace Summaries

```json
{
  "tool": "xray_get_trace_summaries",
  "parameters": {
    "StartTime": "2024-01-15T14:45:00",
    "EndTime": "2024-01-15T15:15:00",
    "FilterExpression": "service(\"checkout-api\") AND responsetime > 2"
  }
}
```

### Reasoning:
| Decision | Ratio

## 7. Benchmark battery

Eight DevOps scenarios. Each one runs the full pipeline and reports per-query reduction. The aggregate establishes whether the skill performs across the deployment domain, not just on the headline scenario.

In [9]:
scenarios = [
    'why is my checkout api slow',
    'show me 5xx errors in the last hour',
    'which downstream service is slow during the latency spike',
    'audit who changed the production database',
    'find the latest backup in s3',
    'ssh into the host without ssh',
    'page the on-call engineer about the queue backlog',
    'why is this getting access denied',
]
print(f'{"reduction":>10}  {"top tool":<34}  query')
print('-' * 100)
reductions = []
for q in scenarios:
    r = lib.select(q, SKILL_DIR / 'sample-aws-tools.json', top_k=5)
    top = r['selected'][0]['name'] if r['selected'] else '-'
    reductions.append(r['reduction_pct'])
    print(f'{r["reduction_pct"]:>9}%  {top:<34}  {q}')
print('-' * 100)
avg = sum(reductions) / len(reductions)
print(f'Average reduction across {len(scenarios)} scenarios: {avg:.1f}%')
assert avg > 50.0, 'Average reduction below 50% — retriever or registry needs work.'
print(f'PASS: avg {avg:.1f}% > 50% floor.')

 reduction  top tool                            query
----------------------------------------------------------------------------------------------------
     88.0%  cloudwatch_logs_insights_query      why is my checkout api slow
     87.7%  cloudwatch_logs_insights_query      show me 5xx errors in the last hour
     81.3%  cloudwatch_logs_insights_query      which downstream service is slow during the latency spike
     82.5%  dynamodb_scan                       audit who changed the production database
     89.8%  s3_list_objects_v2                  find the latest backup in s3
     91.9%  ssm_start_session                   ssh into the host without ssh
     92.3%  sqs_get_queue_attributes            page the on-call engineer about the queue backlog
     92.2%  iam_simulate_principal_policy       why is this getting access denied
----------------------------------------------------------------------------------------------------
Average reduction across 8 scenarios: 88.2%
PASS: avg

## 8. Spike A verdict

Per `PLAN.md` Spike A verification gates:

| Gate | Status |
|------|--------|
| CLI `--help` exits 0 and prints SKILL.md description | PASS (run `python skills/tool-orchestration/rag-mcp-tool-selection/cli.py --help` from project root) |
| CLI returns top-K tools for a query against sample-aws-tools.json | PASS (cell above) |
| Notebook runs top-to-bottom without errors against moto-mocked AWS | PASS (this notebook) |
| Notebook prints measurable token reduction | PASS (above 50% floor) |
| At least one boto3 call executes against mocked AWS and returns shaped data | PASS (queryId returned) |

**Known failure mode documented (see `SKILL.md` Non-Negotiable Verification):** the query `audit who changed the production database` picks `dynamodb_scan` over `cloudtrail_lookup_events` because the simple word-overlap scoring rewards semantic-hook density. Production should swap the retriever for embeddings; the seam is in `lib.score_tool`.

**Spike conclusion:** the chapter primitive → SKILL.md → CLI → notebook → AWS seam works end-to-end. PLAN.md Phase 1 (convert Ch3-Ch7 one chapter at a time) is unblocked.